# 7. `configured out` (node, status)

In [2]:
from common import *

shape, log = show(7)

log shape 82: node | status | configured out
16,027 logs (3.9%) from 256 nodes, 2003-08-05 11:01:31 to 2006-04-26 22:18:00
log 105689 at 2004-06-17 19:32:01: 387046 node-240 node status 1087500721 1 configured out


Don't know what this means.

What happened on the node in the hour before:

In [9]:
around_node = timeline(log.node, log.ts - hour, log.ts)
display(around_node)
previous = around_node[around_node.first_ts < str(log.ts)].iloc[-1]
print(f"the line before it: '{previous.template}', {(log.ts - pd.Timestamp(previous.first_ts)).total_seconds():.0f} seconds earlier")
assert previous.template == "not responding"

,first_ts,last_ts,count,component,state,template
0,2004-06-17 19:31:40,2004-06-17 19:31:40,1,node,status,not responding
1,2004-06-17 19:32:01,2004-06-17 19:32:01,1,node,status,configured out


the line before it: 'not responding', 21 seconds earlier


Ok! so this node wasnt responding and this 'configured out' log probably means the system quickly removed it from the cluster.
#### So the real problem is the 'not responding' message.
Let's make sure that seeing a 'configured out' after a 'not responding' is normal and not a problem of its own.

Every `configured out` of the 30 days before: does a `not responding` on the same node come up earlier?

In [20]:
all_out = as_frame(logs.search(log.ts - 30 * day, log.ts, log_shape_id=shape.log_shape_id, limit=None))
after_not_responding = preceded_by(all_out, NOT_RESPONDING, hour)
display(after_not_responding.value_counts().to_frame("n_logs"))
print(f"{after_not_responding.mean():.0%} of them do")

,n_logs
earlier_ts,
True,502
False,91


85% of them do


Ok so this is common. The real problem is only the 'not responding' log. Is this an isolated node now or something more prevalent?

Nodes that stopped responding in the minutes before our log

In [19]:
down = as_frame(logs.search(log.ts - 10 * minute, log.ts, log_shape_id=NOT_RESPONDING, limit=None))
numbers = sorted(int(node.split("-")[1]) for node in down.node)
print(f"{len(numbers)} nodes, from node-{numbers[0]} to node-{numbers[-1]}, between {down.ts.min().time()} and {down.ts.max().time()}")
assert len(numbers) >= 29 and numbers[-1] - numbers[0] < 32

29 nodes, from node-227 to node-255, between 19:31:39 and 19:31:40


Was it taken care of? (hindsight)
- Boot commands in the 30 minutes after

In [7]:
booted = logs.stats(log.ts, log.ts + 30 * minute, log_shape_id=BOOT)
print(booted)
assert booted.n_nodes == 32

LogStats(n_logs=32, n_nodes=32, first_seen='2004-06-17 19:48:49', last_seen='2004-06-17 20:00:04')


# Verdict: the line is the system's reaction, not the problem. It follows 'not responding' on the same node, here and in most cases.
The problem is that a block of 29 neighbouring nodes stopped responding in the same second. That is serious, and operators booted 32 nodes within the half hour.